In [1]:
try:
    import pyodbc
except ImportError:
    !pip install pyodbc

import pandas as pd

In [2]:
# SQL Server database credentials and connection.

username: str = "sa"
password: str = "ol4_q_7AL"
server: str = 'DESKTOP-6OCN2AJ\SQLEXPRESS'
database: str = "USA_gov_data"

try:
    conn: pyodbc.Connection = pyodbc.connect('DRIVER={SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+ password)
    cursor: pyodbc.Cursor = conn.cursor()
    print("Connection established.")
except pyodbc.Error as e:
    print("Connection failed. More information below: \n" + str(e))

Connection established.


In [3]:
seds_df: pd.DataFrame = pd.read_csv("data/SEDS_Data.csv")
seds_df.head()

,period,seriesId,seriesDescription,stateId,stateDescription,value,unit
0,2023,NUETB,Nuclear energy consumed for electricity genera...,DE,Delaware,0.00,Billion Btu
1,2023,NUETB,Nuclear energy consumed for electricity genera...,FL,Florida,312935.00,Billion Btu
2,2023,NUETB,Nuclear energy consumed for electricity genera...,GA,Georgia,390663.00,Billion Btu
3,2023,NUETD,"Nuclear fuel average price, all sectors",MN,Minnesota,0.77,Dollars per million Btu
4,2023,NUETD,"Nuclear fuel average price, all sectors",MO,Missouri,0.70,Dollars per million Btu


In [4]:
seds_df.describe()

,period,value
count,5000.0,5.000000e+03
mean,2023.0,2.666427e+04
std,0.0,3.393627e+05
min,2023.0,-3.235500e+04
25%,2023.0,5.000000e-01
50%,2023.0,4.514000e+01
75%,2023.0,2.295450e+03
max,2023.0,1.321895e+07


In [5]:
seds_df.describe(include = ['O'])

,seriesId,seriesDescription,stateId,stateDescription,unit
count,5000,5000,5000,5000,5000
unique,158,141,52,52,16
top,NUETB,Biodiesel product supplied portion to the end-...,NV,Nevada,Billion Btu
freq,52,104,103,103,851


In [6]:
seds_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   period             5000 non-null   int64  
 1   seriesId           5000 non-null   object 
 2   seriesDescription  5000 non-null   object 
 3   stateId            5000 non-null   object 
 4   stateDescription   5000 non-null   object 
 5   value              5000 non-null   float64
 6   unit               5000 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 273.6+ KB


In [7]:
script: str = '''
    IF EXISTS (SELECT * FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = 'SEDS_2023')
        DROP TABLE SEDS_2023;
    
    CREATE TABLE SEDS_2023 (
        period INT NOT NULL,
        seriesId VARCHAR(255) NOT NULL,
        seriesDescription VARCHAR(255) NOT NULL,
        stateId VARCHAR(255) NOT NULL,
        stateDescription VARCHAR(255) NOT NULL,
        unit VARCHAR(255) NOT NULL,
        value FLOAT NOT NULL
    );
'''

try:
    cursor.execute(script)
    print("Table created.")
except pyodbc.Error as e:
    print("Table creation failed. More information below: \n" + str(e))

Table created.


In [8]:
# Load data into SQL table.
for index, row in seds_df.iterrows():
    script: str = f'''
    INSERT INTO SEDS_2023 (period, seriesId, seriesDescription, stateId, stateDescription, unit, value)
    VALUES ({row['period']}, '{row['seriesId']}', '{row['seriesDescription']}', '{row['stateId']}', '{row['stateDescription']}', '{row['unit']}', {row['value']});
    '''
    try:
        cursor.execute(script)
    except pyodbc.Error as e:
        print("Data loading failed. More information below: \n" + str(e))
        break

In [9]:
cursor.close()
conn.close()
print("Connection closed.")

Connection closed.
